# 🤖 PENGU Scalping Bot — BingX v4 (BEST VERSION)
**4 celdas. Ejecutar en orden. Solo tocar Celda 1.**

## Mejoras sobre v3
| Aspecto | v3 | v4 |
|---|---|---|
| Tendencia | — | **Supertrend(10,2)** |
| Oscilador | WaveTrend + StochRSI | WaveTrend + StochRSI (mejorados) |
| RSI umbral | sin filtro | **RSI > 55 LONG / < 45 SHORT** |
| Fuerza de tendencia | — | **ADX > 25 + DI direccional** |
| Volumen | 1.1x | **1.5x** |
| Filtro HTF | — | **EMA21/50 en 1h** |
| TP/SL | Fijo % | **Dinámico 2×ATR/4×ATR (R:R 2:1)** |
| Trailing callback | 0.4% | **0.7%** (menos ruido) |

## Lógica de entrada
- **LONG**: Supertrend alcista + WT cruce ↑ desde OS + StochRSI K↑ < 80 + RSI > 55 + ADX > 25 (+DI > -DI) + Vol > 1.5x + HTF 1h alcista/neutro
- **SHORT**: Supertrend bajista + WT cruce ↓ desde OB + StochRSI K↓ > 20 + RSI < 45 + ADX > 25 (-DI > +DI) + Vol > 1.5x + HTF 1h bajista/neutro


In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELDA 1 — Setup y configuración                 ║
# ╚══════════════════════════════════════════════════╝
!pip install pandas numpy requests --quiet

import requests, hmac, hashlib, time, threading
import pandas as pd, numpy as np
from datetime import datetime
import warnings; warnings.filterwarnings('ignore')

# ─── COMPLETAR ANTES DE EJECUTAR ───────────────────
API_KEY    = "TU_API_KEY_AQUI"
SECRET_KEY = "TU_SECRET_KEY_AQUI"
# ───────────────────────────────────────────────────

BASE_URL     = "https://open-api.bingx.com"
SYMBOL       = "PENGU-USDT"
INTERVAL     = "15m"
INTERVAL_HTF = "1h"
LEVERAGE     = 10
CAPITAL_PCT  = 10           # % del balance por trade

# ── TP/SL dinámico por ATR ─────────────────────────
ATR_PERIOD   = 14
SL_ATR_MULT  = 2.0          # SL = 2x ATR
TP_ATR_MULT  = 4.0          # TP = 4x ATR  →  R:R 2:1
MIN_SL_PCT   = 0.004        # SL mínimo 0.4% (evita spread)
MAX_SL_PCT   = 0.018        # SL máximo 1.8% (limita riesgo)

# ── Trailing stop — 3 niveles según fuerza de tendencia ────────────────
TRAIL_ACTIVATE_PCT = 0.010  # activa con +1.0% de ganancia

# Callback por estado de tendencia:
#   FUERTE  → Supertrend ✔ + ADX > 28 + DI confirma  → deja correr (1.5%)
#   NORMAL  → Supertrend ✔ pero ADX moderado           → protección media (0.9%)
#   TIGHT   → Supertrend giró O WaveTrend en contra    → protege ganancias (0.4%)
TRAIL_CB_STRONG    = 0.015
TRAIL_CB_NORMAL    = 0.009
TRAIL_CB_TIGHT     = 0.004
ADX_TRAIL_STRONG   = 28     # ADX mínimo para considerar tendencia fuerte
TRAIL_CHECK_EVERY  = 4      # revisar indicadores cada N × SLEEP_TRAIL (~2 min)

# ── Supertrend ─────────────────────────────────────
ST_PERIOD    = 10
ST_FACTOR    = 2.0

# ── Cipher B / WaveTrend ───────────────────────────
WT_N1 = 10
WT_N2 = 21
WT_OB = 53
WT_OS = -53

# ── Stochastic RSI ─────────────────────────────────
STOCH_RSI_LEN = 14
STOCH_LEN     = 14
STOCH_K       = 3
STOCH_D       = 3
STOCH_OB      = 80
STOCH_OS      = 20

# ── RSI directo ────────────────────────────────────
RSI_LONG     = 55           # RSI mínimo para LONG
RSI_SHORT    = 45           # RSI máximo para SHORT

# ── ADX ────────────────────────────────────────────
ADX_MIN      = 25           # umbral tendencia (era 18 en v1)

# ── Filtro HTF (1h) ────────────────────────────────
HTF_EMA_FAST = 21
HTF_EMA_SLOW = 50

# ── General ────────────────────────────────────────
VOL_MULT     = 1.5          # multiplicador volumen (era 1.1)
COOLDOWN     = 3
KLINES_LIMIT = 200
SLEEP_MAIN   = 60 * 13     # 13 min (vela 15m)
SLEEP_TRAIL  = 30

print('✅ Celda 1 OK')

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELDA 2 — API BingX + Datos de mercado          ║
# ╚══════════════════════════════════════════════════╝

def _ts():
    return int(time.time() * 1000)

def _h():
    return {"X-BX-APIKEY": API_KEY.strip()}

def _signed_url(path, params):
    p = dict(params)
    p["timestamp"] = _ts()
    # BingX verifica la firma contra el query string ordenado alfabéticamente
    qs = "&".join(f"{k}={v}" for k, v in sorted(p.items()))
    sig = hmac.new(SECRET_KEY.strip().encode(), qs.encode(), hashlib.sha256).hexdigest()
    return f"{BASE_URL}{path}?{qs}&signature={sig}"

def api_get(path, params=None, signed=False):
    try:
        if signed:
            url = _signed_url(path, params or {})
            return requests.get(url, headers=_h(), timeout=10).json()
        return requests.get(BASE_URL + path, params=(params or {}), headers=_h(), timeout=10).json()
    except Exception as e:
        print(f"  ⚠️ GET {path}: {e}")
        return {"code": -1}

def api_post(path, params=None):
    try:
        url = _signed_url(path, params or {})
        return requests.post(url, headers=_h(), timeout=10).json()
    except Exception as e:
        print(f"  ⚠️ POST {path}: {e}")
        return {"code": -1}

def api_delete(path, params=None):
    try:
        url = _signed_url(path, params or {})
        return requests.delete(url, headers=_h(), timeout=10).json()
    except Exception as e:
        print(f"  ⚠️ DELETE {path}: {e}")
        return {"code": -1}

def detect_symbol():
    global SYMBOL
    for sym in ["PENGU-USDT", "PENGUSDT", "PENGU_USDT"]:
        r = api_get("/openApi/swap/v2/quote/klines",
                    {"symbol": sym, "interval": "15m", "limit": 3})
        if r.get("code") == 0 and r.get("data"):
            SYMBOL = sym
            print(f"  ✅ Símbolo: {SYMBOL}")
            return True
    print("  ❌ PENGU no encontrado en BingX Futuros")
    return False

def _parse_klines(raw):
    if not raw:
        return None
    if isinstance(raw[0], dict):
        df = pd.DataFrame(raw)
        key_map = {"time": "timestamp", "t": "timestamp",
                   "o": "open", "h": "high", "l": "low", "c": "close", "v": "volume"}
        df = df.rename(columns={k: v for k, v in key_map.items() if k in df.columns})
    else:
        cols = ["timestamp", "open", "high", "low", "close", "volume"]
        df = pd.DataFrame([row[:6] for row in raw], columns=cols)
    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["timestamp"] = pd.to_datetime(
        pd.to_numeric(df["timestamp"], errors="coerce"), unit="ms", errors="coerce")
    df = (df.dropna(subset=["timestamp", "close"])
            .sort_values("timestamp")
            .reset_index(drop=True))
    return df

def get_klines(interval=None, limit=None):
    iv  = interval or INTERVAL
    lim = limit or KLINES_LIMIT
    r = api_get("/openApi/swap/v2/quote/klines",
                {"symbol": SYMBOL, "interval": iv, "limit": lim})
    if not r or r.get("code") != 0:
        print(f"  ❌ Klines ({iv}): {r.get('msg', r)}")
        return None
    df = _parse_klines(r.get("data", []))
    return df if df is not None and len(df) >= 60 else None

def get_price():
    r = api_get("/openApi/swap/v2/quote/premiumIndex", {"symbol": SYMBOL})
    if r.get("code") == 0 and r.get("data"):
        return float(r["data"]["markPrice"])
    df = get_klines()
    return float(df.iloc[-1]["close"]) if df is not None else None

def get_balance():
    r = api_get("/openApi/swap/v2/user/balance", {"currency": "USDT"}, signed=True)
    if r.get("code") == 0:
        try:
            return float(r["data"]["balance"]["availableMargin"])
        except (KeyError, TypeError):
            try:
                return float(r["data"]["availableMargin"])
            except (KeyError, TypeError):
                pass
    print(f"  ❌ Balance: {r.get('msg', r)}")
    return 0.0

def has_open_position():
    r = api_get("/openApi/swap/v2/user/positions", {"symbol": SYMBOL}, signed=True)
    if r.get("code") != 0:
        return False
    return any(abs(float(p.get("positionAmt", 0))) > 0 for p in r.get("data", []))

def set_leverage():
    for side in ("LONG", "SHORT"):
        r = api_post("/openApi/swap/v2/trade/leverage",
                     {"symbol": SYMBOL, "side": side, "leverage": LEVERAGE})
        ok      = r.get("code") == 0
        already = str(r.get("code")) in ("80012", "109107")
        print(f"  {'✅' if ok or already else '❌'} Leverage {LEVERAGE}x {side}"
              f"{' (ya seteado)' if already else ' | ' + r.get('msg', '') if not ok else ''}")

# ── Filtro HTF (1h) ─────────────────────────────────
def get_htf_bias():
    """Retorna 1=alcista, -1=bajista, 0=neutro según EMA21/50 en 1h."""
    df1h = get_klines(interval=INTERVAL_HTF, limit=100)
    if df1h is None or len(df1h) < HTF_EMA_SLOW:
        return 0
    close = df1h["close"]
    ema21 = close.ewm(span=HTF_EMA_FAST, adjust=False).mean()
    ema50 = close.ewm(span=HTF_EMA_SLOW, adjust=False).mean()
    last_close = close.iloc[-1]
    last_e21   = ema21.iloc[-1]
    last_e50   = ema50.iloc[-1]
    if last_close > last_e50 and last_e21 > last_e50:
        return 1
    if last_close < last_e50 and last_e21 < last_e50:
        return -1
    return 0

print('✅ Celda 2 OK')

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELDA 3 — Indicadores + Señales + Órdenes       ║
# ║  Supertrend + WaveTrend + StochRSI + ADX + HTF   ║
# ╚══════════════════════════════════════════════════╝

_bars_since_exit = [999]

# ── Supertrend ──────────────────────────────────────
def calc_supertrend(df, period=ST_PERIOD, factor=ST_FACTOR):
    hi, lo, cl = df["high"].values, df["low"].values, df["close"].values
    n = len(df)

    # ATR con EWM
    tr = np.maximum(hi[1:] - lo[1:],
         np.maximum(np.abs(hi[1:] - cl[:-1]),
                    np.abs(lo[1:]  - cl[:-1])))
    tr = np.concatenate([[hi[0] - lo[0]], tr])
    atr = np.zeros(n)
    atr[0] = tr[0]
    alpha = 2 / (period + 1)
    for i in range(1, n):
        atr[i] = alpha * tr[i] + (1 - alpha) * atr[i - 1]

    hl2    = (hi + lo) / 2
    upper  = hl2 + factor * atr
    lower  = hl2 - factor * atr

    st  = np.zeros(n)
    dir_ = np.zeros(n, dtype=int)
    st[0]   = upper[0]
    dir_[0] = -1

    for i in range(1, n):
        # upper band
        fu = upper[i] if upper[i] < st[i-1] or cl[i-1] > st[i-1] else st[i-1]
        # lower band
        fl = lower[i] if lower[i] > st[i-1] or cl[i-1] < st[i-1] else st[i-1]

        if dir_[i-1] == -1 and cl[i] > st[i-1]:
            dir_[i] = 1;  st[i] = fl
        elif dir_[i-1] == 1 and cl[i] < st[i-1]:
            dir_[i] = -1; st[i] = fu
        elif dir_[i-1] == 1:
            dir_[i] = 1;  st[i] = fl
        else:
            dir_[i] = -1; st[i] = fu

    df["st"]     = st
    df["st_dir"] = dir_
    return df

# ── ADX + DI ────────────────────────────────────────
def calc_adx(df, period=ADX_PERIOD if 'ADX_PERIOD' in dir() else 14):
    hi  = df["high"].values
    lo  = df["low"].values
    cl  = df["close"].values
    n   = len(df)

    up   = np.diff(hi, prepend=hi[0])
    down = -np.diff(lo, prepend=lo[0])
    pdm  = np.where((up > down) & (up > 0), up, 0.0)
    mdm  = np.where((down > up) & (down > 0), down, 0.0)

    tr = np.maximum(hi - lo,
         np.maximum(np.abs(hi - np.roll(cl, 1)),
                    np.abs(lo  - np.roll(cl, 1))))
    tr[0] = hi[0] - lo[0]

    alpha = 1 / ATR_PERIOD
    atr_a = np.zeros(n); atr_a[0] = tr[0]
    pdm_a = np.zeros(n); pdm_a[0] = pdm[0]
    mdm_a = np.zeros(n); mdm_a[0] = mdm[0]
    for i in range(1, n):
        atr_a[i] = alpha * tr[i]  + (1 - alpha) * atr_a[i-1]
        pdm_a[i] = alpha * pdm[i] + (1 - alpha) * pdm_a[i-1]
        mdm_a[i] = alpha * mdm[i] + (1 - alpha) * mdm_a[i-1]

    safe_atr = np.where(atr_a == 0, 1e-10, atr_a)
    pdi = 100 * pdm_a / safe_atr
    mdi = 100 * mdm_a / safe_atr
    dx  = 100 * np.abs(pdi - mdi) / (pdi + mdi + 1e-10)

    adx = np.zeros(n); adx[0] = dx[0]
    for i in range(1, n):
        adx[i] = alpha * dx[i] + (1 - alpha) * adx[i-1]

    df["adx"]      = adx
    df["plus_di"]  = pdi
    df["minus_di"] = mdi
    df["atr14"]    = atr_a
    return df

# ── WaveTrend (Cipher B) ────────────────────────────
def calc_wavetrend(df):
    c  = df["close"]
    hi = df["high"]
    lo = df["low"]
    hlc3 = (hi + lo + c) / 3
    esa  = hlc3.ewm(span=WT_N1, adjust=False).mean()
    d    = (hlc3 - esa).abs().ewm(span=WT_N1, adjust=False).mean()
    d_s  = d.where(d != 0, other=np.nan)
    ci   = (hlc3 - esa) / (0.015 * d_s)
    tci  = ci.fillna(0).ewm(span=WT_N2, adjust=False).mean()
    df["wt1"] = tci
    df["wt2"] = tci.rolling(4).mean()
    df["wt_cross_bull"] = (df["wt1"] > df["wt2"]) & (df["wt1"].shift(1) <= df["wt2"].shift(1))
    df["wt_cross_bear"] = (df["wt1"] < df["wt2"]) & (df["wt1"].shift(1) >= df["wt2"].shift(1))
    return df

# ── Stochastic RSI ──────────────────────────────────
def calc_stoch_rsi(df):
    c     = df["close"]
    delta = c.diff()
    gain  = delta.clip(lower=0).rolling(STOCH_RSI_LEN).mean()
    loss  = (-delta.clip(upper=0)).rolling(STOCH_RSI_LEN).mean()
    rsi   = (100 - 100 / (1 + gain / loss.where(loss != 0, other=np.nan))).fillna(50)
    df["rsi"] = rsi
    rsi_min = rsi.rolling(STOCH_LEN).min()
    rsi_max = rsi.rolling(STOCH_LEN).max()
    rng     = (rsi_max - rsi_min).where((rsi_max - rsi_min) != 0, other=np.nan)
    raw_k   = ((rsi - rsi_min) / rng * 100).fillna(50)
    df["stoch_k"] = raw_k.rolling(STOCH_K).mean()
    df["stoch_d"] = df["stoch_k"].rolling(STOCH_D).mean()
    return df

# ── Volumen ──────────────────────────────────────────
def calc_volume(df):
    df["vol_ma"] = df["volume"].rolling(20).mean()
    return df

# ── Pipeline de indicadores ─────────────────────────
def calculate_indicators(df):
    df = calc_supertrend(df)
    df = calc_adx(df)
    df = calc_wavetrend(df)
    df = calc_stoch_rsi(df)
    df = calc_volume(df)
    return df

# ── TP/SL dinámico por ATR ──────────────────────────
def calc_tp_sl(pos_side, entry, atr):
    sl_dist = float(np.clip(atr * SL_ATR_MULT,
                            entry * MIN_SL_PCT,
                            entry * MAX_SL_PCT))
    tp_dist = sl_dist * (TP_ATR_MULT / SL_ATR_MULT)
    if pos_side == "LONG":
        return round(entry + tp_dist, 7), round(entry - sl_dist, 7)
    return round(entry - tp_dist, 7), round(entry + sl_dist, 7)

# ── Señal ────────────────────────────────────────────
def get_signal(df, htf_bias):
    r     = df.iloc[-2]   # última vela cerrada
    r_ant = df.iloc[-3]   # penúltima (para detectar cruce StochRSI)
    ts    = r["timestamp"].strftime("%H:%M") if pd.notna(r["timestamp"]) else "--:--"

    # Supertrend
    st_bull = int(r["st_dir"]) == 1
    st_bear = int(r["st_dir"]) == -1

    # WaveTrend
    wt_bull   = bool(r["wt_cross_bull"])
    wt_bear   = bool(r["wt_cross_bear"])
    recent_os = (df.iloc[-5:-1]["wt2"] <= WT_OS).any()
    recent_ob = (df.iloc[-5:-1]["wt2"] >= WT_OB).any()

    # Stochastic RSI
    sk, sd         = r["stoch_k"], r["stoch_d"]
    sk_ant, sd_ant = r_ant["stoch_k"], r_ant["stoch_d"]
    stoch_kup = sk > sd and sk_ant <= sd_ant   # cruce K sobre D
    stoch_kdn = sk < sd and sk_ant >= sd_ant   # cruce K bajo D
    sk_not_ob = sk < STOCH_OB                  # K no en sobrecompra extrema al entrar LONG
    sk_not_os = sk > STOCH_OS                  # K no en sobreventa extrema al entrar SHORT

    # RSI directo
    rsi_bull = r["rsi"] > RSI_LONG
    rsi_bear = r["rsi"] < RSI_SHORT

    # ADX
    adx_ok   = r["adx"] > ADX_MIN
    adx_bull = adx_ok and r["plus_di"]  > r["minus_di"]
    adx_bear = adx_ok and r["minus_di"] > r["plus_di"]

    # Volumen
    vol_ok = r["volume"] > r["vol_ma"] * VOL_MULT

    # Cooldown
    cd = _bars_since_exit[0] >= COOLDOWN

    # ── LONG: todas las condiciones deben cumplirse ──
    long_sig = (
        st_bull    and
        wt_bull    and
        recent_os  and
        stoch_kup  and
        sk_not_ob  and
        rsi_bull   and
        adx_bull   and
        vol_ok     and
        htf_bias  >= 0  and  # no operar LONG contra tendencia 1h bajista
        cd
    )

    # ── SHORT: todas las condiciones deben cumplirse ─
    short_sig = (
        st_bear    and
        wt_bear    and
        recent_ob  and
        stoch_kdn  and
        sk_not_os  and
        rsi_bear   and
        adx_bear   and
        vol_ok     and
        htf_bias  <= 0  and  # no operar SHORT contra tendencia 1h alcista
        cd
    )

    # ── Log de estado ───────────────────────────────
    htf_icon = "📈" if htf_bias == 1 else ("📉" if htf_bias == -1 else "➡️")
    st_icon  = "↑" if st_bull else "↓"
    wt_zone  = (f"OS({r['wt2']:.1f})" if r["wt2"] <= WT_OS
                else f"OB({r['wt2']:.1f})" if r["wt2"] >= WT_OB
                else f"({r['wt2']:.1f})")
    print(f"  [{ts}] ST{st_icon} | WT1={r['wt1']:.1f} WT2={wt_zone} | "
          f"StochK={sk:.0f}/D={sd:.0f} | RSI={r['rsi']:.1f} | "
          f"ADX={r['adx']:.1f}{'✔' if adx_ok else '✘'} | "
          f"Vol={'✔' if vol_ok else '✘'} | HTF={htf_icon} | CD={'✔' if cd else '✘'}")

    print(f"  🟢 LONG  {'✅' if long_sig else '─'} "
          f"[ST={'✔' if st_bull else '✘'} WT={'✔' if wt_bull else '✘'} "
          f"OS={'✔' if recent_os else '✘'} Stoch={'✔' if stoch_kup else '✘'} "
          f"RSI={'✔' if rsi_bull else '✘'} ADX={'✔' if adx_bull else '✘'} "
          f"Vol={'✔' if vol_ok else '✘'} HTF={'✔' if htf_bias >= 0 else '✘'}]")
    print(f"  🔴 SHORT {'✅' if short_sig else '─'} "
          f"[ST={'✔' if st_bear else '✘'} WT={'✔' if wt_bear else '✘'} "
          f"OB={'✔' if recent_ob else '✘'} Stoch={'✔' if stoch_kdn else '✘'} "
          f"RSI={'✔' if rsi_bear else '✘'} ADX={'✔' if adx_bear else '✘'} "
          f"Vol={'✔' if vol_ok else '✘'} HTF={'✔' if htf_bias <= 0 else '✘'}]")

    atr = float(r["atr14"])
    px  = float(df.iloc[-1]["close"])
    return long_sig, short_sig, px, atr

# ── Órdenes ──────────────────────────────────────────
def calc_qty(balance, price):
    return int(max(1, round((balance * CAPITAL_PCT / 100 * LEVERAGE) / price)))

def open_order(side, pos_side, qty):
    r = api_post("/openApi/swap/v2/trade/order", {
        "symbol": SYMBOL, "side": side, "positionSide": pos_side,
        "type": "MARKET", "quantity": qty
    })
    print(f"  {'✅' if r.get('code') == 0 else '❌'} {side}/{pos_side} qty={qty} {r.get('msg', '')}")
    return r

def place_tp(pos_side, qty, tp_price):
    close_side = "SELL" if pos_side == "LONG" else "BUY"
    r = api_post("/openApi/swap/v2/trade/order", {
        "symbol": SYMBOL, "side": close_side, "positionSide": pos_side,
        "type": "TAKE_PROFIT_MARKET", "quantity": qty,
        "stopPrice": round(tp_price, 7), "workingType": "MARK_PRICE"
    })
    print(f"  {'✅' if r.get('code') == 0 else '❌'} TP@{tp_price:.7f} {r.get('msg', '')}")

def place_sl(pos_side, qty, sl_price):
    close_side = "SELL" if pos_side == "LONG" else "BUY"
    r = api_post("/openApi/swap/v2/trade/order", {
        "symbol": SYMBOL, "side": close_side, "positionSide": pos_side,
        "type": "STOP_MARKET", "quantity": qty,
        "stopPrice": round(sl_price, 7), "workingType": "MARK_PRICE"
    })
    oid = r.get("data", {}).get("order", {}).get("orderId")
    print(f"  {'✅' if r.get('code') == 0 else '❌'} SL@{sl_price:.7f} {r.get('msg', '')}")
    return oid

def cancel_order(oid):
    if oid:
        api_delete("/openApi/swap/v2/trade/order", {"symbol": SYMBOL, "orderId": oid})

# ── Trailing Stop inteligente (tendencia-aware) ────────────────────────
T = {"on": False, "active": False, "ps": None, "entry": None,
     "qty": None, "sl_id": None, "peak": None, "sl": None}

def reset_trail():
    T.update({"on": False, "active": False, "ps": None, "entry": None,
              "qty": None, "sl_id": None, "peak": None, "sl": None})

def _assess_trend(pos_side):
    """
    Evalúa la fuerza actual de la tendencia y devuelve el callback apropiado.
      TRAIL_CB_STRONG  → tendencia fuerte confirmada (deja correr la posición)
      TRAIL_CB_NORMAL  → tendencia válida pero moderada
      TRAIL_CB_TIGHT   → Supertrend giró o WaveTrend cruza en contra (proteger)
    """
    try:
        df = get_klines()
        if df is None:
            return TRAIL_CB_NORMAL, "sin datos"
        df = calculate_indicators(df)
        row = df.iloc[-2]   # última vela cerrada

        # Supertrend confirma dirección
        st_ok = ((pos_side == "LONG"  and int(row["st_dir"]) == 1) or
                 (pos_side == "SHORT" and int(row["st_dir"]) == -1))

        # ADX fuerte + DI en la dirección correcta
        adx_strong = row["adx"] > ADX_TRAIL_STRONG
        di_ok = ((pos_side == "LONG"  and row["plus_di"]  > row["minus_di"]) or
                 (pos_side == "SHORT" and row["minus_di"] > row["plus_di"]))

        # WaveTrend cruzando en CONTRA (señal de alerta temprana)
        wt_against = ((pos_side == "LONG"  and bool(row["wt_cross_bear"])) or
                      (pos_side == "SHORT" and bool(row["wt_cross_bull"])))

        if not st_ok or wt_against:
            return TRAIL_CB_TIGHT, "tendencia revertida ⚠️"
        if adx_strong and di_ok:
            return TRAIL_CB_STRONG, "tendencia fuerte 💪"
        return TRAIL_CB_NORMAL, "tendencia normal 📊"
    except Exception as e:
        return TRAIL_CB_NORMAL, f"error: {e}"

def _trail_loop():
    check_cnt  = 0
    current_cb = TRAIL_CB_NORMAL   # callback inicial conservador

    while T["on"]:
        time.sleep(SLEEP_TRAIL)

        if not has_open_position():
            print("  📭 Posición cerrada")
            _bars_since_exit[0] = 0
            reset_trail()
            return

        px = get_price()
        if not px:
            continue

        ps, entry = T["ps"], T["entry"]

        # ── Revisar tendencia cada TRAIL_CHECK_EVERY ciclos ───────────
        check_cnt += 1
        if check_cnt >= TRAIL_CHECK_EVERY:
            check_cnt = 0
            new_cb, label = _assess_trend(ps)
            if new_cb != current_cb:
                print(f"  🔁 Trail callback {current_cb*100:.1f}% → {new_cb*100:.1f}% [{label}]")
                current_cb = new_cb
                # Si el callback se apretó y el trail ya está activo,
                # actualizar el SL inmediatamente desde el pico actual
                if T["active"] and T["peak"]:
                    tight_sl = round(T["peak"] * (1 - current_cb) if ps == "LONG"
                                     else T["peak"] * (1 + current_cb), 7)
                    should_move = ((ps == "LONG"  and tight_sl > T["sl"]) or
                                   (ps == "SHORT" and tight_sl < T["sl"]))
                    if should_move:
                        print(f"  🔒 SL ajustado al pico: {T['sl']:.6f} → {tight_sl:.6f}")
                        cancel_order(T["sl_id"])
                        T["sl_id"] = place_sl(ps, T["qty"], tight_sl)
                        T["sl"] = tight_sl

        # ── Activar trailing cuando alcanza el umbral ─────────────────
        if not T["active"]:
            hit = ((ps == "LONG"  and px >= entry * (1 + TRAIL_ACTIVATE_PCT)) or
                   (ps == "SHORT" and px <= entry * (1 - TRAIL_ACTIVATE_PCT)))
            if hit:
                nsl = round(px * (1 - current_cb) if ps == "LONG"
                            else px * (1 + current_cb), 7)
                print(f"  🔄 Trail ON px={px:.6f} SL={nsl:.6f} cb={current_cb*100:.1f}%")
                cancel_order(T["sl_id"])
                T["sl_id"] = place_sl(ps, T["qty"], nsl)
                T.update({"active": True, "peak": px, "sl": nsl})

        # ── Mover SL cuando el precio hace nuevo pico ─────────────────
        else:
            better = ((ps == "LONG" and px > T["peak"]) or
                      (ps == "SHORT" and px < T["peak"]))
            if better:
                T["peak"] = px
                nsl = round(px * (1 - current_cb) if ps == "LONG"
                            else px * (1 + current_cb), 7)
                moved = ((ps == "LONG"  and nsl > T["sl"]) or
                         (ps == "SHORT" and nsl < T["sl"]))
                if moved:
                    print(f"  📈 Trail {T['sl']:.6f} → {nsl:.6f} cb={current_cb*100:.1f}%")
                    cancel_order(T["sl_id"])
                    T["sl_id"] = place_sl(ps, T["qty"], nsl)
                    T["sl"] = nsl

def start_trail(ps, entry, qty, sl_id, sl):
    T.update({"on": True, "active": False, "ps": ps, "entry": entry,
              "qty": qty, "sl_id": sl_id, "peak": entry, "sl": sl})
    threading.Thread(target=_trail_loop, daemon=True).start()

print('✅ Celda 3 OK')

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELDA 4 — INICIAR BOT  ▶                        ║
# ║  Detener: botón ■ o Runtime → Interrupt          ║
# ╚══════════════════════════════════════════════════╝

def run_bot():
    print("═" * 60)
    print("🤖  PENGU SCALPING BOT v4 — BingX Futuros")
    print(f"    {LEVERAGE}x | TP {TP_ATR_MULT}×ATR | SL {SL_ATR_MULT}×ATR | R:R 2:1")
    print(f"    Supertrend({ST_PERIOD},{ST_FACTOR}) + WaveTrend + StochRSI")
    print(f"    ADX > {ADX_MIN} | RSI > {RSI_LONG} / < {RSI_SHORT} | Vol > {VOL_MULT}x")
    print(f"    HTF 1h: EMA{HTF_EMA_FAST}/EMA{HTF_EMA_SLOW}")
    print(f"    Trail +{TRAIL_ACTIVATE_PCT*100:.1f}% | cb fuerte={TRAIL_CB_STRONG*100:.1f}% normal={TRAIL_CB_NORMAL*100:.1f}% tight={TRAIL_CB_TIGHT*100:.1f}%")
    print("═" * 60)

    if not detect_symbol():
        return
    set_leverage()
    reset_trail()
    cycle = 0

    while True:
        try:
            cycle += 1
            print(f"\n{'─' * 60}")
            print(f"⏱  Ciclo {cycle} — {datetime.now().strftime('%H:%M:%S')}")

            if has_open_position():
                print("📌 Posición abierta — trailing activo...")
                time.sleep(SLEEP_MAIN)
                continue

            _bars_since_exit[0] = min(_bars_since_exit[0] + 1, 999)

            # Datos 15m
            df = get_klines()
            if df is None:
                print("  ⚠️ Sin datos 15m — reintentando en 60s")
                time.sleep(60)
                continue

            df = calculate_indicators(df)

            # Filtro HTF 1h
            htf_bias = get_htf_bias()

            long_sig, short_sig, price, atr = get_signal(df, htf_bias)

            balance = get_balance()
            print(f"  💰 Balance: {balance:.2f} USDT")
            if balance < 5:
                print("  ⚠️ Balance insuficiente")
                time.sleep(SLEEP_MAIN)
                continue

            qty = calc_qty(balance, price)

            if long_sig and not short_sig:
                print(f"  🟢 LONG qty={qty} @ {price:.6f}")
                r = open_order("BUY", "LONG", qty)
                if r.get("code") == 0:
                    entry    = get_price() or price
                    tp, isl  = calc_tp_sl("LONG", entry, atr)
                    print(f"  📐 ATR={atr:.6f} | TP={tp:.7f} | SL={isl:.7f}")
                    place_tp("LONG", qty, tp)
                    sl_id = place_sl("LONG", qty, isl)
                    start_trail("LONG", entry, qty, sl_id, isl)
                    _bars_since_exit[0] = 0

            elif short_sig and not long_sig:
                print(f"  🔴 SHORT qty={qty} @ {price:.6f}")
                r = open_order("SELL", "SHORT", qty)
                if r.get("code") == 0:
                    entry    = get_price() or price
                    tp, isl  = calc_tp_sl("SHORT", entry, atr)
                    print(f"  📐 ATR={atr:.6f} | TP={tp:.7f} | SL={isl:.7f}")
                    place_tp("SHORT", qty, tp)
                    sl_id = place_sl("SHORT", qty, isl)
                    start_trail("SHORT", entry, qty, sl_id, isl)
                    _bars_since_exit[0] = 0

            else:
                print("  ⏳ Sin señal")

            time.sleep(SLEEP_MAIN)

        except KeyboardInterrupt:
            print("\n⛔ Bot detenido — revisá posiciones en BingX")
            T["on"] = False
            break
        except Exception as e:
            print(f"  ❌ {e}")
            time.sleep(60)

run_bot()